# Modelos Baseline - Clasificación S&P 500

**Proyecto:** Sistema de Clasificación de Acciones S&P 500

**Grupo 27** - Universidad de Los Andes

---

## Objetivo

Entrenar y comparar 3 modelos baseline con todas las features (17):
1. Logistic Regression (modelo lineal simple)
2. Random Forest (ensemble de árboles)
3. XGBoost (gradient boosting)

Todos los experimentos se registran en MLflow.

In [1]:
import pandas as pd
import numpy as np
import mlflow
import mlflow.sklearn

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, 
    roc_auc_score, confusion_matrix, classification_report
)

import warnings
warnings.filterwarnings('ignore')

## 1. Configuración MLflow

En Databricks, MLflow está pre-instalado. Solo necesitamos configurar el nombre del experimento.

In [2]:
experiment = mlflow.set_experiment("/sp500-baseline-models")

## 2. Carga de Datos

Cargamos los datasets train y test creados en el notebook anterior.

In [ ]:
train = pd.read_parquet('./data/processed/ml_ready/train.parquet')
test = pd.read_parquet('./data/processed/ml_ready/test.parquet')

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")

Train: (20128, 25)
Test:  (5032, 25)


In [4]:
# Separar features (X) y target (Y)
# Excluir Ticker y Date del entrenamiento
feature_cols = [col for col in train.columns if col not in ['Ticker', 'Date', 'Target']]

X_train = train[feature_cols]
y_train = train['Target']

X_test = test[feature_cols]
y_test = test['Target']

print(f"\nFeatures usadas: {len(feature_cols)}")
print(feature_cols)


Features usadas: 22
['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_20', 'SMA_50', 'EMA_12', 'EMA_26', 'RSI_14', 'MACD', 'MACD_signal', 'MACD_diff', 'BB_upper', 'BB_middle', 'BB_lower', 'BB_width', 'ATR_14', 'OBV', 'Returns', 'Volatility_10', 'Volume_change']


## 3. Modelo 1: Logistic Regression

Modelo lineal simple. Sirve como baseline clásico.

In [5]:
with mlflow.start_run(run_name="logistic_regression_baseline"):
    
    # Parámetros por defecto (solo random_state para reproducibilidad)
    params = {
        'random_state': 42
    }
    
    # Entrenar
    lr_model = LogisticRegression(**params)
    lr_model.fit(X_train, y_train)
    
    # Predicciones
    y_pred_lr = lr_model.predict(X_test)
    y_pred_proba_lr = lr_model.predict_proba(X_test)[:, 1]
    
    # Métricas
    accuracy_lr = accuracy_score(y_test, y_pred_lr)
    precision_lr = precision_score(y_test, y_pred_lr)
    recall_lr = recall_score(y_test, y_pred_lr)
    f1_lr = f1_score(y_test, y_pred_lr)
    roc_auc_lr = roc_auc_score(y_test, y_pred_proba_lr)
    
    # Registrar en MLflow
    mlflow.log_params(params)
    mlflow.log_metric("accuracy", accuracy_lr)
    mlflow.log_metric("precision", precision_lr)
    mlflow.log_metric("recall", recall_lr)
    mlflow.log_metric("f1_score", f1_lr)
    mlflow.log_metric("roc_auc", roc_auc_lr)
    
    # Registrar modelo
    mlflow.sklearn.log_model(lr_model, "model")
    
    print("Logistic Regression - Resultados:")
    print(f"Accuracy:  {accuracy_lr:.4f}")
    print(f"Precision: {precision_lr:.4f}")
    print(f"Recall:    {recall_lr:.4f}")
    print(f"F1-Score:  {f1_lr:.4f}")
    print(f"ROC-AUC:   {roc_auc_lr:.4f}")

2025/11/22 22:28:29 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/22 22:28:44 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Logistic Regression - Resultados:
Accuracy:  0.5165
Precision: 0.5165
Recall:    1.0000
F1-Score:  0.6812
ROC-AUC:   0.4915


In [6]:
# Confusion Matrix
cm_lr = confusion_matrix(y_test, y_pred_lr)
print("\nConfusion Matrix:")
print(cm_lr)
print(f"\nTN: {cm_lr[0,0]}, FP: {cm_lr[0,1]}")
print(f"FN: {cm_lr[1,0]}, TP: {cm_lr[1,1]}")


Confusion Matrix:
[[   0 2433]
 [   0 2599]]

TN: 0, FP: 2433
FN: 0, TP: 2599


## 4. Modelo 2: Random Forest

Ensemble de árboles de decisión. Captura relaciones no lineales.

In [7]:
with mlflow.start_run(run_name="random_forest_baseline"):
    
    # Parámetros por defecto (solo random_state para reproducibilidad)
    params = {
        'random_state': 42
    }
    
    # Entrenar
    rf_model = RandomForestClassifier(**params)
    rf_model.fit(X_train, y_train)
    
    # Predicciones
    y_pred_rf = rf_model.predict(X_test)
    y_pred_proba_rf = rf_model.predict_proba(X_test)[:, 1]
    
    # Métricas
    accuracy_rf = accuracy_score(y_test, y_pred_rf)
    precision_rf = precision_score(y_test, y_pred_rf)
    recall_rf = recall_score(y_test, y_pred_rf)
    f1_rf = f1_score(y_test, y_pred_rf)
    roc_auc_rf = roc_auc_score(y_test, y_pred_proba_rf)
    
    # Registrar en MLflow
    mlflow.log_params(params)
    mlflow.log_metric("accuracy", accuracy_rf)
    mlflow.log_metric("precision", precision_rf)
    mlflow.log_metric("recall", recall_rf)
    mlflow.log_metric("f1_score", f1_rf)
    mlflow.log_metric("roc_auc", roc_auc_rf)
    
    # Registrar modelo
    mlflow.sklearn.log_model(rf_model, "model")
    
    print("Random Forest - Resultados:")
    print(f"Accuracy:  {accuracy_rf:.4f}")
    print(f"Precision: {precision_rf:.4f}")
    print(f"Recall:    {recall_rf:.4f}")
    print(f"F1-Score:  {f1_rf:.4f}")
    print(f"ROC-AUC:   {roc_auc_rf:.4f}")

2025/11/22 22:29:08 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/22 22:29:15 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Random Forest - Resultados:
Accuracy:  0.5012
Precision: 0.5133
Recall:    0.6595
F1-Score:  0.5773
ROC-AUC:   0.4975


In [8]:
# Feature Importance
feature_importance_rf = pd.DataFrame({
    'feature': feature_cols,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Features más importantes (Random Forest):")
print(feature_importance_rf.head(10))


Top 10 Features más importantes (Random Forest):
          feature  importance
21  Volume_change    0.062666
19        Returns    0.062073
9          RSI_14    0.057100
16       BB_width    0.056649
12      MACD_diff    0.056632
20  Volatility_10    0.056406
4          Volume    0.056032
18            OBV    0.054338
11    MACD_signal    0.051023
10           MACD    0.050927


## 5. Modelo 3: XGBoost

Gradient boosting. Generalmente el mejor en datos tabulares.

In [9]:
with mlflow.start_run(run_name="xgboost_baseline"):
    
    # Parámetros por defecto (solo random_state para reproducibilidad)
    params = {
        'random_state': 42
    }
    
    # Entrenar
    xgb_model = XGBClassifier(**params)
    xgb_model.fit(X_train, y_train)
    
    # Predicciones
    y_pred_xgb = xgb_model.predict(X_test)
    y_pred_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
    
    # Métricas
    accuracy_xgb = accuracy_score(y_test, y_pred_xgb)
    precision_xgb = precision_score(y_test, y_pred_xgb)
    recall_xgb = recall_score(y_test, y_pred_xgb)
    f1_xgb = f1_score(y_test, y_pred_xgb)
    roc_auc_xgb = roc_auc_score(y_test, y_pred_proba_xgb)
    
    # Registrar en MLflow
    mlflow.log_params(params)
    mlflow.log_metric("accuracy", accuracy_xgb)
    mlflow.log_metric("precision", precision_xgb)
    mlflow.log_metric("recall", recall_xgb)
    mlflow.log_metric("f1_score", f1_xgb)
    mlflow.log_metric("roc_auc", roc_auc_xgb)
    
    # Registrar modelo
    mlflow.sklearn.log_model(xgb_model, "model")
    
    print("XGBoost - Resultados:")
    print(f"Accuracy:  {accuracy_xgb:.4f}")
    print(f"Precision: {precision_xgb:.4f}")
    print(f"Recall:    {recall_xgb:.4f}")
    print(f"F1-Score:  {f1_xgb:.4f}")
    print(f"ROC-AUC:   {roc_auc_xgb:.4f}")

2025/11/22 22:29:17 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/22 22:29:24 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


XGBoost - Resultados:
Accuracy:  0.5093
Precision: 0.5173
Recall:    0.7476
F1-Score:  0.6115
ROC-AUC:   0.5078


In [10]:
# Feature Importance
feature_importance_xgb = pd.DataFrame({
    'feature': feature_cols,
    'importance': xgb_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nTop 10 Features más importantes (XGBoost):")
print(feature_importance_xgb.head(10))


Top 10 Features más importantes (XGBoost):
          feature  importance
3           Close    0.054031
13       BB_upper    0.051284
5          SMA_20    0.050972
18            OBV    0.050454
10           MACD    0.050074
8          EMA_26    0.048592
20  Volatility_10    0.048415
4          Volume    0.048403
21  Volume_change    0.048284
2             Low    0.048095


## 6. Comparación de Modelos

Tabla comparativa de los 3 modelos baseline.

In [11]:
# Crear DataFrame comparativo
resultados = pd.DataFrame({
    'Modelo': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy': [accuracy_lr, accuracy_rf, accuracy_xgb],
    'Precision': [precision_lr, precision_rf, precision_xgb],
    'Recall': [recall_lr, recall_rf, recall_xgb],
    'F1-Score': [f1_lr, f1_rf, f1_xgb],
    'ROC-AUC': [roc_auc_lr, roc_auc_rf, roc_auc_xgb]
})

# Ordenar por ROC-AUC (métrica más robusta para clasificación binaria)
resultados = resultados.sort_values('ROC-AUC', ascending=False).reset_index(drop=True)

print("\nComparación de Modelos Baseline (ordenado por ROC-AUC):")
print(resultados.to_string(index=False))


Comparación de Modelos Baseline (ordenado por ROC-AUC):
             Modelo  Accuracy  Precision   Recall  F1-Score  ROC-AUC
            XGBoost  0.509340   0.517306 0.747595  0.611487 0.507789
      Random Forest  0.501192   0.513327 0.659484  0.577299 0.497518
Logistic Regression  0.516494   0.516494 1.000000  0.681169 0.491462


## Conclusiones Generales

- Los 3 modelos fueron entrenados con parámetros por defecto (solo random_state=42)
- Todos los experimentos están registrados en MLflow
- Los resultados muestran que el problema es más complejo de lo esperado

**Siguiente paso:** Evaluar los 3 modelos con diferentes configuraciones de features (PCA, Feature Selection) en el notebook 02

## 7. Análisis de Resultados

### Observaciones Principales

**1. Rendimiento General:**
- Los tres modelos muestran ROC-AUC alrededor de 0.50 → **No discriminan entre clases** (igual que azar)
- Accuracy ~50-52% apenas mejor que lanzar una moneda
- El problema es más difícil de lo esperado con solo indicadores técnicos

**2. Logistic Regression (ROC-AUC: 0.484):**
- **Peor modelo**: ROC-AUC < 0.5 (peor que aleatorio)
- Recall = 1.0 → predice TODO como "Comprar"
- TN = 0, FN = 0 → No detecta la clase "Vender"
- F1-Score alto (0.68) es **engañoso** por sesgo hacia una clase

**3. Random Forest (ROC-AUC: 0.504):**
- **Mejor modelo** (marginalmente)
- Único con ROC-AUC > 0.5 (aunque apenas)
- Balance razonable: precision 0.52, recall 0.71
- Features clave: Volume_change, Returns, RSI_14

**4. XGBoost (ROC-AUC: 0.498):**
- Prácticamente aleatorio (ROC-AUC ~0.5)
- Mejor balance que Logistic Regression pero sin capacidad predictiva real
- Features: BB_upper, EMA_26, BB_lower

### Por qué ROC-AUC y no F1-Score

**F1-Score es engañoso aquí:**
- Logistic Regression tiene F1=0.68 pero ROC-AUC=0.48
- El F1 alto viene del sesgo (predice todo como "Comprar")
- F1 ignora True Negatives

**ROC-AUC es mejor:**
- Mide discriminación real entre clases
- No se deja engañar por modelos sesgados
- ROC-AUC ~0.5 = modelo aleatorio
- ROC-AUC < 0.5 = peor que aleatorio

### Conclusiones

Los modelos baseline no están capturando patrones:

1. **Features insuficientes:** Los 17 indicadores técnicos no contienen información predictiva
2. **Mercado eficiente:** Movimientos diarios difíciles de predecir con análisis técnico
3. **Ruido vs señal:** Demasiado ruido en movimientos diarios